# POMO Training — HEVRPTW (ARI5002)

**Runtime:** GPU (T4 or A100) — set via Runtime → Change runtime type → GPU

**Steps:**
1. Mount Google Drive and set the project path
2. Install dependencies
3. Run Phase 1 data pipeline (if not already done)
4. Train CVRP model (n=50, 100 epochs)
5. Fine-tune on VRPTW (n=50, 100 epochs)
6. Download checkpoints

In [ ]:
# ── 0. Check GPU ──────────────────────────────────────────────────────────────
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 1. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 2. Project path setup ─────────────────────────────────────────────────────
# Upload the project zip to your Drive, then set the path below.
# Expected Drive structure:
#   MyDrive/POMO_HEVRPTW/
#       ├── pomo/
#       ├── data/
#       ├── configs/
#       ├── train.py
#       ├── evaluate.py
#       ├── requirements.txt
#       └── (raw data files at root)

import os, sys

PROJECT_DIR = '/content/drive/MyDrive/POMO_HEVRPTW'

# If you uploaded a zip, unzip it:
# import zipfile
# with zipfile.ZipFile('/content/drive/MyDrive/POMO_HEVRPTW.zip') as z:
#     z.extractall('/content/')
# PROJECT_DIR = '/content/POMO_HEVRPTW'

sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# ── 3. Install dependencies ───────────────────────────────────────────────────
# torch is pre-installed on Colab; install the rest
%pip install -q openpyxl pyyaml folium ortools tqdm

In [ ]:
# ── 4. Build Phase 1 data pipeline (skip if processed/ already exists) ────────
import os
PROCESSED = os.path.join(PROJECT_DIR, 'data', 'processed')

if not os.path.exists(os.path.join(PROCESSED, 'real_instances.pkl')):
    print('Building data pipeline...')
    from data.fleet_parser import build_fleet_registry
    from data.instance_builder import build_instances
    from data.baseline_extractor import extract_baselines

    build_fleet_registry(save=True)
    instances = build_instances(save=True)
    extract_baselines(save=True)
    print(f'Done. {len(instances)} instances.')
else:
    import pickle
    with open(os.path.join(PROCESSED, 'real_instances.pkl'), 'rb') as f:
        instances = pickle.load(f)
    print(f'Loaded {len(instances)} existing instances.')

In [ ]:
# ── 5a. CVRP Training (n=50, 100 epochs) ─────────────────────────────────────
# Expected time: ~50-70 min on T4, ~22-35 min on A100
# AMP (mixed precision) is now automatic in Trainer when CUDA is available.

import yaml, torch
from pomo.trainer import Trainer

with open('configs/default.yaml') as f:
    config = yaml.safe_load(f)

# B=32: flat batch = 32×50 = 1600 → fits T4 comfortably with AMP
# (B=64 → 3200 → OOM on T4; B=32 keeps same total data, 80 steps/epoch)
config['training']['n_nodes'] = 50
config['training']['epochs'] = 100
config['training']['batch_size'] = 32
config['training']['synthetic_instances_per_epoch'] = 2560  # 80 batches/epoch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {device}')

trainer_cvrp = Trainer(config, device=device, mode='cvrp')
trainer_cvrp.train()

In [ ]:
# ── 5b. VRPTW Training (n=50, 100 epochs) ────────────────────────────────────
# config already has batch_size=32 and n_nodes=50 from the cell above.
# AMP is automatic in Trainer on CUDA.

trainer_vrptw = Trainer(config, device=device, mode='vrptw')

# Option B: fine-tune from CVRP checkpoint (shared encoder weights)
# CVRP checkpoint has input_dim=3, VRPTW needs input_dim=5.
# Load only shared weights (encoder body + decoder), skip input_proj.
# import os
# cvrp_ckpt = torch.load('checkpoints/best_cvrp.pt', map_location=device)
# state = cvrp_ckpt['model_state']
# state = {k: v for k, v in state.items() if 'input_proj' not in k}
# trainer_vrptw.model.load_state_dict(state, strict=False)
# print('Fine-tuning from CVRP checkpoint (shared weights loaded)')

trainer_vrptw.train()

In [ ]:
# ── 5d. Re-finetune with lower LR (if 5c made things worse) ──────────────────
# The first fine-tune used lr=1e-4 (same as pre-training) which caused
# catastrophic forgetting. This cell uses lr=1e-5 (10x lower) for gentler
# adaptation. Load the ORIGINAL best_vrptw.pt, not the degraded finetune.
#
# Expected time: ~15-20 min on T4.

import importlib, pomo.trainer, pomo.model, pomo.env
importlib.reload(pomo.env); importlib.reload(pomo.model); importlib.reload(pomo.trainer)
from pomo.trainer import Trainer
import yaml, torch

with open('configs/default.yaml') as f:
    config = yaml.safe_load(f)

config['training']['batch_size'] = 32
config['training']['epochs'] = 20           # fewer epochs with lower LR
config['training']['lr'] = 1e-5            # 10x lower than pre-training
config['training']['synthetic_instances_per_epoch'] = 2560

device = 'cuda' if torch.cuda.is_available() else 'cpu'

trainer_ft2 = Trainer(config, device=device, mode='finetune')
trainer_ft2.load_checkpoint('checkpoints/best_vrptw.pt')   # start from VRPTW, not finetune
print('Re-finetuning with lr=1e-5 from best_vrptw.pt...')

trainer_ft2.train()
print('Done. Saved as checkpoints/best_finetune.pt')

In [ ]:
# ── 5c. Fine-tune on real-statistics VRPTW instances (30 epochs) ─────────────
# WHY: Training used demand_norm~[0.02,0.18] and TW width~[72,168]min.
#      Real DHL instances have demand_norm~0.007 (10x smaller) and 48% all-day TW.
#      The model returns to depot too often on real data because it thinks capacity
#      runs out (it doesn't). Fine-tuning on matching statistics closes this gap.
#
# Load the VRPTW checkpoint, then fine-tune for 30 epochs.
# Expected time: ~15-20 min on T4.

import yaml, torch, os
from pomo.trainer import Trainer

with open('configs/default.yaml') as f:
    config = yaml.safe_load(f)

config['training']['batch_size'] = 32
config['training']['epochs'] = 30
config['training']['synthetic_instances_per_epoch'] = 2560  # 80 batches/epoch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

trainer_ft = Trainer(config, device=device, mode='finetune')

# Load the VRPTW checkpoint as starting point
trainer_ft.load_checkpoint('checkpoints/best_vrptw.pt')
print('Fine-tuning from VRPTW checkpoint...')

trainer_ft.train()

# Fine-tuned checkpoint is saved as checkpoints/best_finetune.pt
print('Done. Download checkpoints/best_finetune.pt')

In [ ]:
# ── 6. Quick evaluation on synthetic benchmarks ───────────────────────────────
import torch, yaml
from pomo.model import POMOModel
from pomo.trainer import generate_cvrp_batch
from pomo.inference import greedy_rollout, _augment_coords

with open('configs/default.yaml') as f:
    config = yaml.safe_load(f)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load best CVRP checkpoint
model = POMOModel(config).to(device)
ckpt = torch.load('checkpoints/best_cvrp.pt', map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()

# Evaluate on 128 synthetic n=50 instances with 8x augmentation
B, n, cap = 128, 50, 50
node_xy, demand, demand_norm, cap_t, dist_matrix = generate_cvrp_batch(B, n, cap, device)
nf = torch.cat([node_xy, demand_norm.unsqueeze(-1)], dim=-1)

best_rewards = torch.full((B,), -float('inf'), device=device)
with torch.no_grad():
    for aug_idx in range(8):
        aug_xy = _augment_coords(node_xy, aug_idx)
        aug_nf = torch.cat([aug_xy, demand_norm.unsqueeze(-1)], dim=-1)
        rewards, _ = greedy_rollout(model, aug_nf, demand, cap_t, dist_matrix, device)
        best_rewards = torch.maximum(best_rewards, rewards.max(dim=1).values)

tour_lengths = -best_rewards
print(f'n=50 CVRP (8x aug): mean={tour_lengths.mean():.4f}, '
      f'min={tour_lengths.min():.4f}, max={tour_lengths.max():.4f}')
print('Reference (POMO paper, n=50): ~10.37')

In [ ]:
# ── 7. Real instance evaluation ───────────────────────────────────────────────
# Evaluate VRPTW model on real Istanbul DHL instances

import pickle, csv
from pomo.model import POMOModel
from pomo.inference import solve

with open('configs/default.yaml') as f:
    config = yaml.safe_load(f)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load VRPTW model
model_vrptw = POMOModel(config).to(device)
model_vrptw.set_input_dim(5)
ckpt = torch.load('checkpoints/best_vrptw.pt', map_location=device)
model_vrptw.load_state_dict(ckpt['model_state'])
model_vrptw.eval()

with open('data/processed/real_instances.pkl', 'rb') as f:
    instances = pickle.load(f)

# Load baseline
baseline = {}
with open('data/processed/baseline_metrics.csv') as f:
    for row in csv.DictReader(f):
        key = (row['route'], row['depot'], row['date'])
        baseline[key] = float(row['total_dist_km'])

results = []
for inst in instances[:50]:  # evaluate first 50 to save time; remove [:50] for full eval
    key = (inst['route'], inst['depot_name'],
            str(inst['date'].date()) if hasattr(inst['date'], 'date') else str(inst['date']))
    base_dist = baseline.get(key)

    with torch.no_grad():
        tour, pomo_dist = solve(model_vrptw, inst, use_augmentation=True,
                                augmentation_count=8, device=device)
    results.append({
        'route': inst['route'], 'depot': inst['depot_name'],
        'vehicle_type': inst['vehicle_type'], 'is_ev': inst['is_electric'],
        'n_nodes': inst['n_nodes'],
        'pomo_km': pomo_dist, 'baseline_km': base_dist,
        'reduction_pct': 100*(base_dist - pomo_dist)/base_dist if base_dist else None,
    })

reductions = [r['reduction_pct'] for r in results if r['reduction_pct'] is not None]
print(f'Evaluated {len(results)} instances')
print(f'Mean distance reduction: {sum(reductions)/len(reductions):.2f}%')
print(f'POMO mean dist: {sum(r["pomo_km"] for r in results)/len(results):.2f} km')

In [ ]:
# ── 8. Download checkpoints ───────────────────────────────────────────────────
# Option A: files are already in Google Drive (checkpoints/ folder)
# Just download from Drive UI.

# Option B: download directly to local machine
# from google.colab import files
# files.download('checkpoints/best_cvrp.pt')
# files.download('checkpoints/best_vrptw.pt')

import os
for f in sorted(os.listdir('checkpoints/')):
    size = os.path.getsize(f'checkpoints/{f}') / 1e6
    print(f'  checkpoints/{f}  ({size:.1f} MB)')

In [ ]:
# ── 9. Synthetic HEVRPTW Benchmark: POMO vs OR-Tools vs Nearest Neighbour ─────
# PRIMARY EVALUATION for the paper (replaces real-instance eval as main result).
#
# Evaluates both CVRP and VRPTW models on synthetic instances (n=20, 50).
# For n=20: also runs OR-Tools (multi-vehicle, 15s/instance) → optimality gap.
# For n=50: POMO only (OR-Tools is too slow).
#
# Actual results from Mac MPS run:
#   CVRP  n=20: POMO greedy 4.47, OR-Tools 4.85 → POMO BEATS OR-Tools by 7.8%
#   CVRP  n=50: POMO greedy 9.14, NN 12.47       → POMO beats NN by 27%
#   VRPTW n=20: POMO greedy 4.75, OR-Tools 5.04  → POMO BEATS OR-Tools by 5.7%
#   VRPTW n=50: POMO greedy 9.70
#   EV range: range=2.0 units → +8.8% tour length; range≥3.0 → no impact
#
# VRPTW note: training T=480 min but travel times ≤1.41 → TW never bound during
# training → VRPTW model ≈ CVRP with 5-dim input (see distribution analysis).
#
# Runtime: ~5 min on T4 (OR-Tools runs on CPU regardless of GPU)

%pip install -q ortools

import importlib, sys, os
for mod in list(sys.modules.keys()):
    if mod.startswith('pomo') or mod.startswith('eval'):
        del sys.modules[mod]

sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

from eval.benchmark_eval import (
    gen_cvrp_batch, gen_vrptw_batch,
    nearest_neighbour_cvrp,
    solve_ortools_cvrp, solve_ortools_vrptw,
    _pomo_cvrp, _pomo_vrptw,
    load_model, ev_range_analysis,
    print_summary_table, save_csv,
)
import numpy as np, yaml, torch, time

with open('configs/default.yaml') as f:
    config = yaml.safe_load(f)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on: {device}')

print('\nLoading checkpoints...')
model_cvrp  = load_model('checkpoints/best_cvrp.pt',  config, device)
model_vrptw = load_model('checkpoints/best_vrptw.pt', config, device)

N_INSTANCES  = 100
ORT_LIMIT    = 15     # seconds per OR-Tools instance
ORT_N        = 50     # instances sent to OR-Tools (n=20 only)
all_rows     = []

for problem, n_list in [('cvrp', [20, 50]), ('vrptw', [20, 50])]:
    model = model_cvrp if problem == 'cvrp' else model_vrptw
    print(f'\n{"═"*60}\n  {problem.upper()} Benchmark\n{"═"*60}')

    for n in n_list:
        print(f'\n  n={n}, {N_INSTANCES} instances')

        if problem == 'cvrp':
            coords, demand, cap, dm = gen_cvrp_batch(N_INSTANCES, n, base_seed=0)
        else:
            coords, demand, cap, dm, tw, T = gen_vrptw_batch(N_INSTANCES, n, base_seed=1000)

        nn_lens = None
        if problem == 'cvrp':
            nn_lens = [nearest_neighbour_cvrp(coords[b], demand[b], cap)
                       for b in range(N_INSTANCES)]
            print(f'    Nearest Neighbour:    {np.mean(nn_lens):.4f}')

        pomo_fn = _pomo_cvrp if problem == 'cvrp' else _pomo_vrptw
        pomo_args = (model, coords, demand, cap, dm, device)
        vrptw_kwargs = dict(tw_np=tw, T=T) if problem == 'vrptw' else {}

        pomo_lens = (_pomo_cvrp(model, coords, demand, cap, dm, device, use_aug=False)
                     if problem == 'cvrp'
                     else _pomo_vrptw(model, coords, demand, cap, dm, tw, T, device, use_aug=False))
        print(f'    POMO (greedy):        {np.mean(pomo_lens):.4f}')

        aug_lens = (_pomo_cvrp(model, coords, demand, cap, dm, device, use_aug=True)
                    if problem == 'cvrp'
                    else _pomo_vrptw(model, coords, demand, cap, dm, tw, T, device, use_aug=True))
        print(f'    POMO (8× aug):        {np.mean(aug_lens):.4f}')

        ort_lens = []
        if n <= 20:
            print(f'    OR-Tools ({ORT_N} inst, {ORT_LIMIT}s each)...', end='', flush=True)
            t0 = time.time()
            for b in range(ORT_N):
                res = (solve_ortools_cvrp(coords[b], demand[b], cap, ORT_LIMIT)
                       if problem == 'cvrp'
                       else solve_ortools_vrptw(coords[b], demand[b], cap, tw[b], T, ORT_LIMIT))
                if res is not None:
                    ort_lens.append(res)
            print(f' done ({time.time()-t0:.0f}s, {len(ort_lens)}/{ORT_N} solved)')
            print(f'    OR-Tools:             {np.mean(ort_lens):.4f}')
            gap     = 100*(np.mean(pomo_lens[:ORT_N]) - np.mean(ort_lens))/np.mean(ort_lens)
            aug_gap = 100*(np.mean(aug_lens[:ORT_N])  - np.mean(ort_lens))/np.mean(ort_lens)
            print(f'    Gap (greedy vs ORT):  {gap:+.2f}%  (negative = POMO better)')
            print(f'    Gap (8×aug  vs ORT):  {aug_gap:+.2f}%')

        all_rows.append({
            'problem': problem.upper(), 'n': n,
            'nn_mean':          round(float(np.mean(nn_lens)),   4) if nn_lens  else None,
            'pomo_mean':        round(float(np.mean(pomo_lens)), 4),
            'pomo_aug_mean':    round(float(np.mean(aug_lens)),  4),
            'ort_mean':         round(float(np.mean(ort_lens)),  4) if ort_lens else None,
            'ort_n_solved':     len(ort_lens),
            'pomo_gap_pct':     round(100*(np.mean(pomo_lens[:ORT_N])-np.mean(ort_lens))/np.mean(ort_lens), 2) if ort_lens else None,
            'pomo_aug_gap_pct': round(100*(np.mean(aug_lens[:ORT_N]) -np.mean(ort_lens))/np.mean(ort_lens), 2) if ort_lens else None,
        })

ev_range_analysis(model_cvrp, n=20, B=100, device=device,
                  range_vals=(None, 5.0, 3.0, 2.0))

save_csv(all_rows, 'data/processed/benchmark_results.csv')
print_summary_table(all_rows)

# Generate figures
from viz.benchmark_plots import load_results, plot_method_comparison, plot_ev_sensitivity, plot_pomo_vs_nn
import os
fig_dir = 'viz/figures'
os.makedirs(fig_dir, exist_ok=True)
rows = load_results('data/processed/benchmark_results.csv')
plot_method_comparison(rows, fig_dir)
plot_ev_sensitivity(fig_dir)
plot_pomo_vs_nn(rows, fig_dir)
print('\nFigures saved to viz/figures/')

## Tips

**GPU selection:** A100 trains ~3x faster than T4. Use Colab Pro for A100 access.

**Mixed precision (optional speedup):**
```python
from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()
with autocast():
    rewards, log_prob_sum = rollout_flat(...)
    loss = -(advantage * log_prob_sum).mean()
scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()
```
Add this inside `train_epoch()` in `pomo/trainer.py` for ~1.5-2x speedup on A100.

**Expected training times (n=50, B=64, 100 epochs):**
| Hardware | Time/epoch | Total |
|----------|-----------|-------|
| T4 GPU | ~50-60s | ~85-100 min |
| A100 GPU | ~20-25s | ~35-40 min |
| M2 Mac MPS | ~30s | ~50 min |

**n=50 CVRP benchmarks (expected after 100 epochs):**
- Mean tour length ≈ 10.37 (POMO paper reports 10.37 for n=50)
- With 8x augmentation: ~10.20-10.35